In [4]:
import numpy as np
import random
import csv

In [37]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)



def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

#  calcul a erorii cross-entropy
def cross_entropy_loss(y_true, y_predicted):
    return -np.sum(y_true * np.log(y_predicted + 1e-9)) / y_true.shape[0]


def initialize_population(population_size, num_weights):
    return [np.random.uniform(-1, 1, num_weights) for _ in range(population_size)]

# evaluare fitness pentru un individ
def fitness_eval(individ, X, y, structure):
    weights, biases = decode_individ(individ, structure)
    predictions = forward_pass(X, weights, biases)
    loss = cross_entropy_loss(y, predictions)
    return -loss  # fitness-ul este negativul erorii




# decodificare individ
def decode_individ(individ, structure):
    weights = []
    biases = []
    index = 0
    for i in range(len(structure) - 1):
        intrare_dim = structure[i]
        iesire_dim = structure[i + 1]
        weight_count = intrare_dim * iesire_dim
        bias_count = iesire_dim
        
        weights.append(individ[index:index + weight_count].reshape(intrare_dim, iesire_dim))
        index += weight_count
        biases.append(individ[index:index + bias_count])
        index += bias_count
    return weights, biases




# trecere prin retea
def forward_pass(X, weights, biases):
    layer_intrare = X
    
    for i in range(len(weights) - 1):
        layer_iesire = sigmoid(np.dot(layer_intrare, weights[i]) + biases[i])
        layer_intrare = layer_iesire
        
    final_output = softmax(np.dot(layer_intrare, weights[-1]) + biases[-1])
    return final_output






# selectia parintilorr pe baza fitnessului in turneu
def select_parents(population, fitness, tournament_size=3):
    parents = []
    for _ in range(2):
        tournament = random.sample(range(len(population)), tournament_size)
        best = max(tournament, key=lambda i: fitness[i])
        
        parents.append(population[best])
    return parents

# incrucisare intre 2 parinti
def crossover(parinte1, parinte2, crossover_rate=0.7):
    if random.random() < crossover_rate:
        point = random.randint(1, len(parinte1) - 1)
        
        copil1 = np.concatenate((parinte1[:point], parinte2[point:]))
        
        copil2 = np.concatenate((parinte2[:point], parinte1[point:]))
        return copil1, copil2
    return parinte1.copy(), parinte2.copy()




# mutarea unui individ
def mutate(individual, mutation_rate=0.1):
    for i in range(len(individual)):
        if random.random() < mutation_rate:
            
            individual[i] += np.random.normal(0, 0.1)
    return individual

def train_evolutionary(X, y, structure, population_size=50, generations=50, mutation_rate=0.2):
    nr_weights = sum(structure[i] * structure[i + 1] + structure[i + 1] for i in range(len(structure) - 1))
    
    population = initialize_population(population_size, nr_weights)
    
    for generation in range(generations):
        fitness = [fitness_eval(individ, X, y, structure) for individ in population]
        new_population = []

        
        for _ in range(population_size // 2):
            parinte1, parinte2 = select_parents(population, fitness)
            copil1, copil2 = crossover(parinte1, parinte2)
            
            new_population.append(mutate(copil1, mutation_rate))
            new_population.append(mutate(copil2, mutation_rate))

        population = new_population
        best_fitness = max(fitness)
        
        print(f"generatia {generation + 1}: cel mai bun fitness = {best_fitness}")

    best_individ = population[np.argmax(fitness)]
    weights, biases = decode_individ(best_individ, structure)
    return weights, biases



def evaluate_performance(predictions, labels):
    predicted_classes = np.argmax(predictions, axis=1) + 1
    correct = np.sum(predicted_classes == labels)
    
    accuracy = correct / len(labels)
    print(f"acuratetea retelei: {accuracy * 100:.2f}%")
    return accuracy

In [28]:
def load_data(filename):
    data = []
    labels = []
    
    
    with open(filename, 'r') as file:
        reader = csv.reader(file, delimiter=';')
        next(reader) 
        for row in reader:
            data.append(list(map(float, row[:-1])))
            labels.append(int(row[-1]))
            
    return np.array(data), np.array(labels)

In [29]:
data, wine_quality_labels=load_data("winequality-red.csv");
print(data[0:4])
print(len(data[0]))
print(len(wine_quality_labels))
print(wine_quality_labels[0:4])

[[7.400e+00 7.000e-01 0.000e+00 1.900e+00 7.600e-02 1.100e+01 3.400e+01
  9.978e-01 3.510e+00 5.600e-01 9.400e+00]
 [7.800e+00 8.800e-01 0.000e+00 2.600e+00 9.800e-02 2.500e+01 6.700e+01
  9.968e-01 3.200e+00 6.800e-01 9.800e+00]
 [7.800e+00 7.600e-01 4.000e-02 2.300e+00 9.200e-02 1.500e+01 5.400e+01
  9.970e-01 3.260e+00 6.500e-01 9.800e+00]
 [1.120e+01 2.800e-01 5.600e-01 1.900e+00 7.500e-02 1.700e+01 6.000e+01
  9.980e-01 3.160e+00 5.800e-01 9.800e+00]]
11
1599
[5 5 5 6]


In [38]:
    nr_clas = 10
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    print(y)
    # structura retelei
    structure = [11,6,7,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure)

    
    # Afișare rezultate finale
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, labels)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
generatia 1: cel mai bun fitness = -1.6663909614768955
generatia 2: cel mai bun fitness = -1.5650542422442424
generatia 3: cel mai bun fitness = -1.4701918032002013
generatia 4: cel mai bun fitness = -1.3389200800699277
generatia 5: cel mai bun fitness = -1.3561206801115868
generatia 6: cel mai bun fitness = -1.3051913536066986
generatia 7: cel mai bun fitness = -1.2904899356671615
generatia 8: cel mai bun fitness = -1.2770188201356254
generatia 9: cel mai bun fitness = -1.2573348280586087
generatia 10: cel mai bun fitness = -1.2525798601083817
generatia 11: cel mai bun fitness = -1.2472885591059133
generatia 12: cel mai bun fitness = -1.2346915541324586
generatia 13: cel mai bun fitness = -1.221760841310529
generatia 14: cel mai bun fitness = -1.2204142202891919
generatia 15: cel mai bun fitness = -1.212434121862342
generatia 16: c

0.425891181988743

In [19]:
print(len(predictions))

1599


In [40]:
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    print(y)
    # structura retelei
    structure = [11,3,4,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure,population_size=100,generations=30, mutation_rate=0.7)

    
    # Afișare rezultate finale
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, labels)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
generatia 1: cel mai bun fitness = -1.7203734326524789
generatia 2: cel mai bun fitness = -1.6099436522334867
generatia 3: cel mai bun fitness = -1.370320190204352
generatia 4: cel mai bun fitness = -1.374880096004563
generatia 5: cel mai bun fitness = -1.3942324167438154
generatia 6: cel mai bun fitness = -1.2962143568714204
generatia 7: cel mai bun fitness = -1.3511904427663668
generatia 8: cel mai bun fitness = -1.3072148601214595
generatia 9: cel mai bun fitness = -1.2821629565110553
generatia 10: cel mai bun fitness = -1.2674505210009772
generatia 11: cel mai bun fitness = -1.2675318353845046
generatia 12: cel mai bun fitness = -1.251948101934115
generatia 13: cel mai bun fitness = -1.2432998952068723
generatia 14: cel mai bun fitness = -1.235571885827149
generatia 15: cel mai bun fitness = -1.2276827332905762
generatia 16: cel

0.38711694809255787

In [43]:
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    # structura retelei
    structure = [11,7,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure,population_size=75,generations=50, mutation_rate=0.4)

    
    # Afișare rezultate finale
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, labels)

generatia 1: cel mai bun fitness = -1.4510015672382806
generatia 2: cel mai bun fitness = -1.3956959238233095
generatia 3: cel mai bun fitness = -1.3763985603154258
generatia 4: cel mai bun fitness = -1.4008953917296143
generatia 5: cel mai bun fitness = -1.3432789446146447
generatia 6: cel mai bun fitness = -1.3532023767889323
generatia 7: cel mai bun fitness = -1.3100463591530893
generatia 8: cel mai bun fitness = -1.3068861315653728
generatia 9: cel mai bun fitness = -1.273942762974009
generatia 10: cel mai bun fitness = -1.2683486357586948
generatia 11: cel mai bun fitness = -1.270086261513732
generatia 12: cel mai bun fitness = -1.2468663309559425
generatia 13: cel mai bun fitness = -1.2295607995758073
generatia 14: cel mai bun fitness = -1.2291867782751602
generatia 15: cel mai bun fitness = -1.2112167508588885
generatia 16: cel mai bun fitness = -1.2114295347518365
generatia 17: cel mai bun fitness = -1.2018589450073407
generatia 18: cel mai bun fitness = -1.1987802978732387
gen

0.4640400250156348

In [44]:
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    print(y)
    # structura retelei
    structure = [11,5,5,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure,population_size=100,generations=400, mutation_rate=0.5)

    
    # Afișare rezultate finale
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, labels)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
generatia 1: cel mai bun fitness = -1.595011068095177
generatia 2: cel mai bun fitness = -1.5816448225486452
generatia 3: cel mai bun fitness = -1.5438317677800764
generatia 4: cel mai bun fitness = -1.4815336088379318
generatia 5: cel mai bun fitness = -1.4495697599746857
generatia 6: cel mai bun fitness = -1.4251419089178265
generatia 7: cel mai bun fitness = -1.3696803859327003
generatia 8: cel mai bun fitness = -1.3460017029332996
generatia 9: cel mai bun fitness = -1.3270466521718596
generatia 10: cel mai bun fitness = -1.293602860825394
generatia 11: cel mai bun fitness = -1.277738466693289
generatia 12: cel mai bun fitness = -1.2485572119918553
generatia 13: cel mai bun fitness = -1.2257966525969093
generatia 14: cel mai bun fitness = -1.2339220916422051
generatia 15: cel mai bun fitness = -1.220288730751458
generatia 16: cel

generatia 145: cel mai bun fitness = -1.0953984816045836
generatia 146: cel mai bun fitness = -1.0963117125668804
generatia 147: cel mai bun fitness = -1.0898329354055523
generatia 148: cel mai bun fitness = -1.100712926514479
generatia 149: cel mai bun fitness = -1.1002429652968548
generatia 150: cel mai bun fitness = -1.1010082216494828
generatia 151: cel mai bun fitness = -1.0994128423295138
generatia 152: cel mai bun fitness = -1.1025028575460618
generatia 153: cel mai bun fitness = -1.0990083589702095
generatia 154: cel mai bun fitness = -1.106252107256034
generatia 155: cel mai bun fitness = -1.109428798136056
generatia 156: cel mai bun fitness = -1.1012100941623308
generatia 157: cel mai bun fitness = -1.102211293522325
generatia 158: cel mai bun fitness = -1.1139391153966898
generatia 159: cel mai bun fitness = -1.109246612583916
generatia 160: cel mai bun fitness = -1.1102138448300087
generatia 161: cel mai bun fitness = -1.102958442477003
generatia 162: cel mai bun fitness = 

generatia 291: cel mai bun fitness = -1.054284736708337
generatia 292: cel mai bun fitness = -1.0541971413279838
generatia 293: cel mai bun fitness = -1.0620503187179569
generatia 294: cel mai bun fitness = -1.0523399022975637
generatia 295: cel mai bun fitness = -1.051549199299712
generatia 296: cel mai bun fitness = -1.0672625459552076
generatia 297: cel mai bun fitness = -1.0518376274476047
generatia 298: cel mai bun fitness = -1.0678748818883737
generatia 299: cel mai bun fitness = -1.0641174805022768
generatia 300: cel mai bun fitness = -1.057827197315105
generatia 301: cel mai bun fitness = -1.0560123336213147
generatia 302: cel mai bun fitness = -1.0475692285577651
generatia 303: cel mai bun fitness = -1.0578961513506735
generatia 304: cel mai bun fitness = -1.0479876865205573
generatia 305: cel mai bun fitness = -1.0541208605768322
generatia 306: cel mai bun fitness = -1.0587644659932143
generatia 307: cel mai bun fitness = -1.0782496763890335
generatia 308: cel mai bun fitness

0.4283927454659162